In [1]:
import warnings
warnings.filterwarnings('ignore')
import polars as pl
import polars.selectors as cs
import numpy as np
import math
import seaborn as sns
import os 
import re
import matplotlib.pyplot as plt
import gzip
import matplotlib.colors as mcolors
from scipy import stats
pl.Config.set_fmt_str_lengths(50)
pl.Config().set_tbl_rows(2000)
sns.set_style(style='white')
warnings.filterwarnings('ignore')

1. Validate that the luciferase assay recapitulates ccMPRA-identified activity and provides baseline measurements for each promoter.
-- 20 strong enhancers (10 encode and 10 non-encode annotated) + the promoters alone
-- 20 strong silencers + the promoters alone
= 80 sequences
2. Test the promoter-dependent activity.
-- 3 CREs that act as enhancers or silencers based on the promoter + the promoters alone
-- 3 CREs that act as enhancer for 1 promoter and no effect on the other promoter + the promoters alone
-- 3 CREs that act as silencer for 1 promoter and no effect on the other promoter + the promoters alone
= 36 sequences
3. Then the reviewer 2 was worried about the effect of the length variation, coordinates variation and distance from H3k27ac peaks so I was thinking:
-- Reuse 5 strong enhancers and 5 strong silencers that we tested in point 1 and test 2 additional lengths for them (20). No need to redo the promoter alone.
-- Among those 10 select 5 and test the effect of 2 shifts of the position as compared to the H3k27ac peak (10). No need to redo the promoter alone.
-- Among the promoters we used in point 1, we select 5 and we try an other length for them, either alone either with their CRE (fixe length, same then point 1) (10).
= 40 sequences

## Filtering

Filtering on:
- both parts at least 100 bp
- CRE is not an encode PLS
- promoter contains TSS


In [11]:
cd = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/"
data = pl.read_csv(os.path.join(cd,"results/MPRA_analysis/CMPRA5/labeled_data_promoteroa_OA.tsv"), separator="\t")
data = data.filter(pl.col("right_bin").str.contains("null").not_()).rename({"OE": "CRE", "nr_reads": "nr_barcodes", "dist": "distance", "interaction": "ENCODE_labels"})
data = data.filter(pl.col("label") != "other - other")
data = data.filter(pl.col("any_tss") == "yes")
data = data.with_columns(
	CRE_length = pl.col("CRE").str.split("-").list.get(2).cast(pl.Int64) - pl.col("CRE").str.split("-").list.get(1).cast(pl.Int64),
	promoter_length = pl.col("promoter").str.split("-").list.get(2).cast(pl.Int64) - pl.col("promoter").str.split("-").list.get(1).cast(pl.Int64)
)
data = data.filter((pl.col("CRE_length") >= 100) & (pl.col("promoter_length") >= 100))
data = data.with_columns(pl.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("PLS")) & pl.any_horizontal(cs.matches("screen").is_null())))
			 			.then(pl.lit("PLS - undefined"))
						.when(pl.all_horizontal(pl.any_horizontal(cs.matches("screen").str.contains("ELS")) & pl.any_horizontal(cs.matches("screen").is_null())))
						.then(pl.lit("ELS - undefined"))
						 .when(pl.all_horizontal(cs.matches("screen").is_null())).then(pl.lit("undefined")).otherwise(pl.col("ENCODE_labels"))
						 .alias("ENCODE_labels"))
data = data.filter(pl.col("ENCODE_labels").str.contains("PLS - PLS").not_()).select(~cs.matches("left|right|tss|Val|std"))

## Enhancers and silencers

In [12]:
silencers = pl.concat([data.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score").head(200),
						data.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score").head(200)])
enhancers = pl.concat([data.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score", descending=True).head(200),
						data.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score", descending=True).head(200)])

## Multi interacting CREs

In [62]:
multi_prom = data.filter(pl.col("promoter").n_unique().over("CRE") >= 2) \
	.filter((pl.col("z_score").max().over("CRE") > 1.5) | (pl.col("z_score").min().over("CRE") < -1.5)) \
	.with_columns(activity_difference = np.abs(pl.col("z_score").max().over("CRE") - pl.col("z_score").min().over("CRE")))\
		.sort("activity_difference", descending=True)

multi_prom = multi_prom.filter(pl.col("CRE").is_in(multi_prom.select("CRE").unique(maintain_order=True).head(100)["CRE"]))

## Getting bin order of original sequences for enhancers and silencers

In [7]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/results/mprasnakeflow/results/assigned_bcs_part1and2.tsv.gz' | cut -f -2,4,6,8,10,12,14 | awk 'NR > 1 && NF >=5' | cut -f 2 > temp.seqids.tsv

In [8]:
pl.concat([silencers, enhancers]).select(pl.col("CRE")).write_csv("CREs.temp.ids.tsv", include_header=False)
pl.concat([silencers, enhancers]).select(pl.col("promoter")).write_csv("promoter.temp.ids.tsv", include_header=False)

In [9]:
seq_ids = "temp.seqids.tsv"
cre_ids = "CREs.temp.ids.tsv"
prom_ids = "promoter.temp.ids.tsv"

In [10]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.temp.ids.tsv | grep -Fwf promoter.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.tsv

In [11]:
# Check uniqueness
! echo $(cut -f -2 temp.actualbins.tsv | sort | uniq |  wc -l) $(wc -l < temp.actualbins.tsv)
! echo $(wc -l < CREs.temp.ids.tsv) # looks like there are some incorrect bins

1205 1205
800


In [7]:
actual_bins = pl.read_csv("temp.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"}) 

In [8]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr1-11805849-11806206-+""","""chr1-11842196-11842568-.""",12
"""chr1-12392815-12393089-.""","""chr1-12618206-12618597--""",14
"""chr1-12393090-12393210-.""","""chr1-12618206-12618597--""",13
"""chr1-12618206-12618597--""","""chr1-12350334-12350578-.""",17
"""chr1-12618206-12618597--""","""chr1-12565474-12565661-.""",33


In [132]:
! grep "chr1-44988261-44988729-." temp.actualbins.tsv

chr1-44988261-44988729-.	chr1-45012148-45012412-+	m84066_240516_024507_s2/116200304/ccs,m84066_240516_024507_s2/117968625/ccs,m84066_240516_024507_s2/119738712/ccs,m84066_240516_024507_s2/123341587/ccs,m84066_240516_024507_s2/125175647/ccs,m84066_240516_024507_s2/134353403/ccs,m84066_240516_024507_s2/161482149/ccs,m84066_240516_024507_s2/197003724/ccs,m84066_240516_024507_s2/236719141/ccs,m84066_240516_024507_s2/264769070/ccs,m84066_240516_024507_s2/53546331/ccs,m84066_240516_024507_s2/67899501/ccs,m84066_240516_024507_s2/69010563/ccs,m84066_240623_102831_s3/101520839/ccs,m84066_240623_102831_s3/146478915/ccs,m84066_240623_102831_s3/157025067/ccs,m84066_240623_102831_s3/169610701/ccs,m84066_240623_102831_s3/239473830/ccs,m84066_240623_102831_s3/83168331/ccs,m84066_240623_122801_s1/17567107/ccs,m84066_240623_122801_s1/247466350/ccs,m84066_240623_122801_s1/37097751/ccs,m84066_240623_122801_s1/37687676/ccs
chr1-45012148-45012412-+	chr1-44988261-44988729-.	m84066_240516_024507_s2/34406993/

In [ ]:
# --------- Code if we want to try multiple bin orders -----------

# Get the silencers with the bin order in which they were sequenced, and additionally in a CRE - promoter order
silencers_with_bin_order = silencers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
silencers_with_bin_order = pl.concat([silencers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])

enhancers_with_bin_order = enhancers.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
enhancers_with_bin_order = pl.concat([enhancers_with_bin_order, 
									  actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])
enhancers_with_bin_order.height


51

In [13]:
# --------- Code if we want to test CRE - promoter order only -----------

silencers_with_bin_order = actual_bins.select(pl.exclude("column_3")).join(silencers, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)

enhancers_with_bin_order = actual_bins.select(pl.exclude("column_3")).join(enhancers, left_on=["left_bin", "right_bin"], right_on=["CRE", "promoter"], coalesce = False)
silencers_with_bin_order.head()

left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64
"""chr1-12392815-12393089-.""","""chr1-12618206-12618597--""",-1.377689,5,1,"""target - other""","""PLS - undefined""",225449.5,"""DHRS3""","""downregulating""","""chr1-12618206-12618597--""","""chr1-12392815-12393089-.""",0.097648,-2.695592,274,391
"""chr1-12393090-12393210-.""","""chr1-12618206-12618597--""",-1.377082,5,1,"""target - other""","""PLS - undefined""",225251.5,"""DHRS3""","""downregulating""","""chr1-12618206-12618597--""","""chr1-12393090-12393210-.""",0.097648,-2.694482,120,391
"""chr1-42956006-42956133-.""","""chr1-42959066-42959704--""",-1.522336,6,2,"""target - other""","""PLS - ELS""",3315.5,"""SLC2A1""","""no effect""","""chr1-42959066-42959704--""","""chr1-42956006-42956133-.""",-0.592023,-1.968959,127,638
"""chr1-44988261-44988729-.""","""chr1-45012148-45012412-+""",-0.708816,15,3,"""positive - other""","""PLS - undefined""",23785.0,"""UROD""","""downregulating""","""chr1-45012148-45012412-+""","""chr1-44988261-44988729-.""",0.655515,-3.764063,468,264
"""chr1-45017871-45018224-.""","""chr1-45012148-45012412-+""",-0.354499,7,2,"""positive - other""","""PLS - undefined""",5767.5,"""UROD""","""downregulating""","""chr1-45012148-45012412-+""","""chr1-45017871-45018224-.""",0.655515,-2.786533,353,264


In [14]:
silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").height, enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").height
#silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").height, enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").height

(125, 113)

In [55]:
final_silencers = pl.concat([silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score").head(10),
						silencers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score").head(10)])
final_enhancers = pl.concat([enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - undefined").sort("z_score", descending=True).head(10),
						enhancers_with_bin_order.filter(pl.col("ENCODE_labels") == "PLS - ELS").sort("z_score", descending=True).head(10)])
final_enhancers

## Getting bin order of original sequences for multi-promoter CREs

In [63]:
multi_prom.select(pl.col("CRE")).write_csv("CREs.multiprom.temp.ids.tsv", include_header=False)
multi_prom.select(pl.col("promoter")).write_csv("promoter.multiprom.temp.ids.tsv", include_header=False)
cre_ids = "CREs.multiprom.temp.ids.tsv"
prom_ids = "promoter.multiprom.temp.ids.tsv"

In [64]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.multiprom.temp.ids.tsv | grep -Fwf promoter.multiprom.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.multiprom.actualbins.tsv

In [65]:
actual_bins = pl.read_csv("temp.multiprom.actualbins.tsv", separator="\t", has_header=False).rename({"column_1": "left_bin", "column_2": "right_bin"})

In [34]:
actual_bins.with_columns(column_3 = pl.col("column_3").str.split(",").list.len()).head()

left_bin,right_bin,column_3
str,str,u32
"""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",16
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",16
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",70
"""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",67
"""chr14-35122081-35122903--""","""chr14-35115858-35116161-.""",41


Dont forget: there are more sequences in the bin, that are not used in the end, so which are not counted in nr_seqs

In [36]:
multi_prom.filter(pl.col("CRE") == "chr14-20955217-20955500-.")

logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108


In [92]:
multi_prom_with_binorder = multi_prom.select(pl.col("CRE").alias("left_bin"), pl.col("promoter").alias("right_bin"), pl.all())
multi_prom_with_binorder = pl.concat([multi_prom_with_binorder, 
									  actual_bins.select(pl.exclude("column_3")).join(multi_prom, left_on=["left_bin", "right_bin"], right_on=["promoter", "CRE"], coalesce = False)])
multi_prom_with_binorder.sort("activity_difference", descending=True).head()

left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
"""chr14-20955217-20955500-.""","""chr14-20688098-20688455-+""",-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108
"""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
"""chr20-17963417-17963734-.""","""chr20-17968096-17968866-+""",-1.259062,8,3,"""target - other""","""PLS - ELS""",4905.5,"""MGME1""","""downregulating""","""chr20-17968096-17968866-+""","""chr20-17963417-17963734-.""",-0.157991,-2.240839,317,770,4.326135
"""chr20-17963417-17963734-.""","""chr20-17968096-17968866--""",0.005819,5,2,"""target - other""","""PLS - ELS""",4905.5,"""SNX5""","""upregulating""","""chr20-17968096-17968866--""","""chr20-17963417-17963734-.""",-0.887512,2.085295,317,770,4.326135


## Orientation of CREs

In [16]:
pl.concat([silencers_with_bin_order, enhancers_with_bin_order]).select(pl.col("CRE")).write_csv("CREs.withbinorder.temp.ids.tsv", include_header=False)
pl.concat([silencers_with_bin_order, enhancers_with_bin_order]).select(pl.col("promoter")).write_csv("promoter.withbinorder.temp.ids.tsv", include_header=False)

In [17]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.withbinorder.temp.ids.tsv | grep -Fwf promoter.withbinorder.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.withbinorder.tsv

In [19]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3"-.\t"$4"\t"$5}' | grep -Fwf CREs.withbinorder.temp.ids.tsv | \
		grep -Fwf <(cut -f 3 temp.actualbins.withbinorder.tsv | sed 's/,/\n/g') > left_bins_with_orientation.temp.tsv

!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3"-.\t"$4"\t"$5}' | grep -Fwf CREs.withbinorder.temp.ids.tsv | \
		grep -Fwf <(cut -f 3 temp.actualbins.withbinorder.tsv | sed 's/,/\n/g') > right_bins_with_orientation.temp.tsv


In [34]:
# Am I losing sequences here?

!paste <(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}' ) \
		<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}')| awk -v OFS="\t" '{print $1,$3,$2,$4}' | \
		grep -Fwf <(cut -f -2 temp.actualbins.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) | sort  \
			> temp.actualbins.withorientation.part1.tsv

In [35]:
!paste <(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}' ) \
		<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}')| awk -v OFS="\t" '{print $3,$1,$4,$2}' | \
		grep -Fwf <(cut -f -2 temp.actualbins.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) | sort  \
			> temp.actualbins.withorientation.part2.tsv

In [42]:
!cat temp.actualbins.withorientation.part1.tsv temp.actualbins.withorientation.part2.tsv | sort > temp.actualbins.withorientation.tsv

In [82]:
!awk '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4}' temp.actualbins.withorientation.tsv | grep -Fwf promoter.withbinorder.temp.ids.tsv | sort | uniq -c > temp.promoter.orientation.counts.tsv

In [48]:
!awk '{ \
	key = $2 FS $3; \
	count[key] += $1; \
	row_count[key]++; \
	lines[key,row_count[key]] = $0; \
	val6[key,row_count[key]] = $6; \
	val7[key,row_count[key]] = $7; \
	cnt[key,row_count[key]] = $1 \
} \
END { \
	for (k in count) { \
		if (row_count[k] == 1) { \
			print val6[k,1], val7[k,1] \
		} else if (row_count[k] == 2) { \
			if (cnt[k,1] / count[k] > 0.5) { \
				print val6[k,1], val7[k,1] \
			} else if (cnt[k,2] / count[k] > 0.5) { \
				print val6[k,2], val7[k,2] \
			} else { \
				print "ambiguous", "ambiguous" ; \
			} \
		} \
	} \
} \
' temp.promoter.orientation.counts.tsv > temp.finalbins.tsv # final number of promoters with unique orientation

			# } else { \
			# 	print count[k], lines[k,1], "ambiguous", "ambiguous" ; \
			# 	print count[k], lines[k,2], "ambiguous", "ambiguous" ; \
			# }\

In [60]:
final_bins = pl.read_csv("temp.finalbins.tsv", separator=" ", has_header=False)\
.rename({"column_1": "CRE_withorientation", "column_2": "promoter"}).with_columns(
	CRE = pl.col("CRE_withorientation").str.replace("--","-.") \
		.str.replace("-\+","-."))
enhancers_with_bin_order.join(final_bins, on=["promoter", "CRE"]) \
.sort("z_score").filter((pl.col("ENCODE_labels").str.contains("ELS"))
& (pl.col("z_score") > 2)).height

silencers_with_bin_order.join(final_bins, on=["promoter", "CRE"]) \
.filter((pl.col("ENCODE_labels").str.contains("ELS").not_())
& (pl.col("z_score") < -2)).height


28

## Orientation of CREs multi promoter

In [71]:
multi_prom_with_binorder.select(pl.col("CRE")).write_csv("CREs.multiprom.withbinorder.temp.ids.tsv", include_header=False)
multi_prom_with_binorder.select(pl.col("promoter")).write_csv("promoter.multiprom.withbinorder.temp.ids.tsv", include_header=False)

In [72]:
!awk -v FS="\t" '{print $1"-"$2"-"$3"-"$4"\t"$5"-"$6"-"$7"-"$8"\t"$9}' \
	<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/promoteroabinaware_bins_all.filt_OA.bed.gz') \
		| grep -v "\-\-\-" | grep -Fwf CREs.multiprom.withbinorder.temp.ids.tsv | grep -Fwf promoter.multiprom.withbinorder.temp.ids.tsv | grep -Fwf <(sed 's/>//' temp.seqids.tsv) \
			> temp.actualbins.multiprom.withbinorder.tsv

In [74]:
!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3"-.\t"$4"\t"$5}' | grep -Fwf CREs.multiprom.withbinorder.temp.ids.tsv | \
		grep -Fwf <(cut -f 3 temp.actualbins.multiprom.withbinorder.tsv | sed 's/,/\n/g') > left_multiprom_bins_with_orientation.temp.tsv

!zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3"-.\t"$4"\t"$5}' | grep -Fwf CREs.multiprom.withbinorder.temp.ids.tsv | \
		grep -Fwf <(cut -f 3 temp.actualbins.multiprom.withbinorder.tsv | sed 's/,/\n/g') > right_multiprom_bins_with_orientation.temp.tsv


In [75]:
# Am I losing sequences here?

!paste <(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}' ) \
		<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}')| awk -v OFS="\t" '{print $1,$3,$2,$4}' | \
		grep -Fwf <(cut -f -2 temp.actualbins.multiprom.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) | sort  \
			> temp.actualbins.multiprom.withorientation.part1.tsv

In [76]:
!paste <(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_left_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}' ) \
		<(zcat '/data/cephfs-2/unmirrored/groups/kircher/MPRA/CaptureCMPRA/mpra_capture_flow/results/CMPRA5/binned/singlebinssmalleroverlap_right_bins_OA.bed.gz' \
	| awk -v FS="\t" '{print $1"-"$2"-"$3,$5}')| awk -v OFS="\t" '{print $3,$1,$4,$2}' | \
		grep -Fwf <(cut -f -2 temp.actualbins.multiprom.withbinorder.tsv | sed 's/-\.//g' | sed 's/-+//g' | sed 's/--//g' ) | sort  \
			> temp.actualbins.multiprom.withorientation.part2.tsv

In [77]:
!cat temp.actualbins.multiprom.withorientation.part1.tsv temp.actualbins.multiprom.withorientation.part2.tsv | sort > temp.actualbins.multiprom.withorientation.tsv

In [80]:
!awk '{print $1,$2,$3,$4,$1"-"$3,$2"-"$4}' temp.actualbins.multiprom.withorientation.tsv | grep -Fwf promoter.multiprom.withbinorder.temp.ids.tsv | sort | uniq -c > temp.promoter.multiprom.orientation.counts.tsv

In [83]:
!awk '{ \
	key = $2 FS $3; \
	count[key] += $1; \
	row_count[key]++; \
	lines[key,row_count[key]] = $0; \
	val6[key,row_count[key]] = $6; \
	val7[key,row_count[key]] = $7; \
	cnt[key,row_count[key]] = $1 \
} \
END { \
	for (k in count) { \
		if (row_count[k] == 1) { \
			print val6[k,1], val7[k,1] \
		} else if (row_count[k] == 2) { \
			if (cnt[k,1] / count[k] > 0.5) { \
				print val6[k,1], val7[k,1] \
			} else if (cnt[k,2] / count[k] > 0.5) { \
				print val6[k,2], val7[k,2] \
			} else { \
				print "ambiguous", "ambiguous" ; \
			} \
		} \
	} \
} \
' temp.promoter.multiprom.orientation.counts.tsv > temp.multiprom.finalbins.tsv # final number of promoters with unique orientation

			# } else { \
			# 	print count[k], lines[k,1], "ambiguous", "ambiguous" ; \
			# 	print count[k], lines[k,2], "ambiguous", "ambiguous" ; \
			# }\

In [87]:
multi_prom_with_binorder.head()

left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64
"""chr14-20955217-20955500-.""","""chr14-20688098-20688455-+""",-1.086703,11,2,"""target - other""","""PLS - undefined""",267082.0,"""ANG""","""no effect""","""chr14-20688098-20688455-+""","""chr14-20955217-20955500-.""",-1.379707,0.916286,283,357,4.60108
"""chr14-20955217-20955500-.""","""chr14-20890827-20891165-+""",0.618555,11,1,"""negative - other""","""PLS - undefined""",64362.5,"""RNASE3""","""upregulating""","""chr14-20890827-20891165-+""","""chr14-20955217-20955500-.""",-1.287388,5.517367,283,338,4.60108
"""chr20-17963417-17963734-.""","""chr20-17968096-17968866-+""",-1.259062,8,3,"""target - other""","""PLS - ELS""",4905.5,"""MGME1""","""downregulating""","""chr20-17968096-17968866-+""","""chr20-17963417-17963734-.""",-0.157991,-2.240839,317,770,4.326135
"""chr20-17963417-17963734-.""","""chr20-17968096-17968866--""",0.005819,5,2,"""target - other""","""PLS - ELS""",4905.5,"""SNX5""","""upregulating""","""chr20-17968096-17968866--""","""chr20-17963417-17963734-.""",-0.887512,2.085295,317,770,4.326135
"""chr17-75252859-75253070-.""","""chr17-75261404-75261785--""",1.227543,5,2,"""target - other""","""PLS - undefined""",8630.0,"""GGA3""","""no effect""","""chr17-75261404-75261785--""","""chr17-75252859-75253070-.""",0.602591,1.722083,211,381,4.307408


In [91]:
final_multiprom_bins = pl.read_csv("temp.multiprom.finalbins.tsv", separator=" ", has_header=False)\
.rename({"column_1": "CRE_withorientation", "column_2": "promoter"}).with_columns(
	CRE = pl.col("CRE_withorientation").str.replace("--","-.") \
		.str.replace("-\+","-."))
multi_prom_with_binorder.join(final_multiprom_bins, on=["promoter", "CRE"])\
.sort("activity_difference", descending=True).filter(pl.col("right_bin") == pl.col("promoter"))


left_bin,right_bin,logFC,nr_barcodes,nr_seqs,label,ENCODE_labels,distance,target_genes,effect,promoter,CRE,promoter_only,z_score,CRE_length,promoter_length,activity_difference,CRE_withorientation
str,str,f64,i64,i64,str,str,f64,str,str,str,str,f64,f64,i64,i64,f64,str
"""chr14-35115858-35116161-.""","""chr14-35121748-35122080--""",0.419135,30,6,"""target - other""","""PLS - undefined""",5904.5,"""PPP2R3C""","""no effect""","""chr14-35121748-35122080--""","""chr14-35115858-35116161-.""",0.046627,1.151496,303,332,3.946451,"""chr14-35115858-35116161--"""
"""chr5-134335047-134335177-.""","""chr5-134370784-134371130-+""",1.081665,5,1,"""target - other""","""PLS - undefined""",35845.0,"""UBE2B""","""no effect""","""chr5-134370784-134371130-+""","""chr5-134335047-134335177-.""",0.146108,1.267931,130,346,3.903103,"""chr5-134335047-134335177-+"""
"""chr14-35375308-35375460-.""","""chr14-35404489-35405088--""",1.004291,7,1,"""target - other""","""PLS - undefined""",29404.5,"""NFKBIA""","""upregulating""","""chr14-35404489-35405088--""","""chr14-35375308-35375460-.""",0.171183,2.799543,152,599,3.658537,"""chr14-35375308-35375460-+"""
"""chr20-45465166-45465284-.""","""chr20-45415817-45416355-+""",0.345329,8,1,"""positive - other""","""PLS - undefined""",49139.0,"""PIGT""","""no effect""","""chr20-45415817-45416355-+""","""chr20-45465166-45465284-.""",0.289312,0.193045,118,538,3.417972,"""chr20-45465166-45465284-+"""
"""chr14-34959309-34959415-.""","""chr14-34875340-34875553--""",0.453758,14,2,"""target - other""","""PLS - undefined""",83915.5,"""BAZ1A""","""no effect""","""chr14-34875340-34875553--""","""chr14-34959309-34959415-.""",-0.471866,1.482918,106,213,3.30441,"""chr14-34959309-34959415-+"""
"""chr14-35124371-35124677-.""","""chr14-35122081-35122903--""",-1.072217,16,4,"""target - other""","""PLS - undefined""",2032.0,"""PPP2R3C""","""downregulating""","""chr14-35122081-35122903--""","""chr14-35124371-35124677-.""",-0.224263,-2.179876,306,822,3.149666,"""chr14-35124371-35124677-+"""
"""chr6-116296603-116296747-.""","""chr6-116279963-116280517--""",1.581271,14,2,"""target - other""","""PLS - undefined""",16435.0,"""TSPYL1""","""no effect""","""chr6-116279963-116280517--""","""chr6-116296603-116296747-.""",-0.422487,1.771529,144,554,3.089504,"""chr6-116296603-116296747--"""
"""chr20-35457427-35457600-.""","""chr20-35742180-35742625--""",1.396575,5,1,"""target - other""","""PLS - undefined""",284889.0,"""RBM39""","""downregulating""","""chr20-35742180-35742625--""","""chr20-35457427-35457600-.""",2.328104,-2.424646,173,445,3.020836,"""chr20-35457427-35457600-+"""
"""chr19-40029094-40029219-.""","""chr19-39971166-39971563-+""",-1.452996,6,1,"""positive - other""","""PLS - undefined""",57792.0,"""PSMC4""","""no effect""","""chr19-39971166-39971563-+""","""chr19-40029094-40029219-.""",-0.615902,-1.580321,125,397,2.991902,"""chr19-40029094-40029219--"""


## Write to files

In [41]:
#silencers_with_bin_order.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_silencers.tsv"), separator="\t")
#enhancers_with_bin_order.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_enhancers.tsv"), separator="\t")
multi_prom_with_binorder.write_csv(os.path.join(cd,"results/luciferase_design/luciferase_design_dual_function.tsv"), separator="\t")